# Entrega 2 — Experimentación Metodológica y Análisis Principal
## Predicción de Deserción Estudiantil: Invarianza entre PreTec21 y Tec21
**Equipo 7 · Desarrollo de Aplicaciones Avanzadas de Ciencias Computacionales**

---
Este notebook implementa experimentación sistemática con tres aproximaciones metodológicas
(K-Means, Regresión Logística, Random Forest), comparación de resultados, y documentación
de experimentos negativos.

> **Diferencia con `exploration.ipynb`:** este notebook parte del dataset ya imputado
> (`data/dataset_imputed.csv`) generado por `data/preprocessing/02_imputation_pipeline.ipynb`.
> El preprocesamiento de missingness ya fue aplicado; aquí solo se realiza ingeniería
> de variables, encoding y split temporal.

### Estructura
1. Setup e importaciones
2. Carga del dataset imputado
3. Exploración inicial (distribución, desbalance, correlaciones)
4. Ingeniería de variables y split temporal
5. K-Means Clustering (perfiles de riesgo)
6. Regresión Logística (línea base interpretable)
7. Random Forest (captura de interacciones no lineales)
8. Comparación sistemática
9. Experimentos negativos

## 1. Setup e Importaciones

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings, time, os
from pathlib import Path
from scipy import stats

from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.model_selection import (
    StratifiedKFold, cross_validate, RandomizedSearchCV, learning_curve,
    train_test_split
)
from sklearn.metrics import (
    roc_auc_score, recall_score, f1_score, precision_score,
    confusion_matrix, roc_curve, precision_recall_curve,
    ConfusionMatrixDisplay, classification_report,
    silhouette_score, davies_bouldin_score
)
from sklearn.inspection import permutation_importance
from scipy.stats import randint, uniform

warnings.filterwarnings('ignore')

# ── Globals ──────────────────────────────────────────────────────────────────
SEED        = 42
TRAIN_GENS  = ['AD14', 'AD15', 'AD16', 'AD17', 'AD18']   # PreTec21
TEST_GENS   = ['AD19', 'AD20']                              # Tec21
TARGET      = 'retention'
K_CLUSTERS  = 4
N_BOOTSTRAP = 1000

np.random.seed(SEED)
DATA_DIR    = Path('../data')
IMG_DIR     = Path('images')
IMG_DIR.mkdir(exist_ok=True)

print("Librerías cargadas correctamente.")
print(f"  sklearn  : {__import__('sklearn').__version__}")
print(f"  pandas   : {pd.__version__}")
print(f"  numpy    : {np.__version__}")

## 2. Carga del Dataset Imputado

El archivo `dataset_imputed.csv` fue generado por `data/preprocessing/02_imputation_pipeline.ipynb`
y contiene:
- Registros de nivel universitario únicamente (AD14–AD20)
- Valores faltantes normalizados (strings "No information", "Does not apply" → NaN → imputados)
- 7 columnas indicadoras de missingness original
- Columna `regime` (0 = PreTec21, 1 = Tec21)

In [ ]:
IMPUTED_CSV = DATA_DIR / 'dataset_imputed.csv'

t0 = time.time()
df_raw = pd.read_csv(IMPUTED_CSV, low_memory=False)
print(f"✓ dataset_imputed.csv cargado en {time.time()-t0:.1f}s  →  {df_raw.shape}")
print(f"\nColumnas ({df_raw.shape[1]}):")
print(df_raw.columns.tolist())

print(f"\nDistribución por generación:")
print(df_raw['generation'].value_counts().sort_index().to_string())

# Verificar regímenes
pretec21 = df_raw[df_raw['generation'].isin(TRAIN_GENS)]
tec21    = df_raw[df_raw['generation'].isin(TEST_GENS)]
print(f"\nPreTec21 (AD14–AD18): {len(pretec21):,}")
print(f"Tec21    (AD19–AD20): {len(tec21):,}")

# Indicadores de missingness original
indicator_cols = [
    'took_admission_test', 'first_gen_present', 'parents_edu_present',
    'has_life_activities', 'has_socioeconomic_data',
    'has_social_lag_data', 'has_zone_data'
]
print(f"\nIndicadores de missingness disponibles: {indicator_cols}")
print(df_raw[indicator_cols].mean().round(3).to_string())

## 3. Exploración Inicial

In [ ]:
# ── Desbalance de clases ──────────────────────────────────────────────────────
counts = df_raw[TARGET].value_counts()
pcts   = df_raw[TARGET].value_counts(normalize=True) * 100
print(pd.DataFrame({
    'Etiqueta': ['Desertó (0)', 'Retuvo (1)'],
    'N'       : [counts[0], counts[1]],
    '%'       : [pcts[0].round(2), pcts[1].round(2)]
}).to_string(index=False))

fig, ax = plt.subplots(figsize=(4, 3))
ax.bar(['Retuvo (1)', 'Desertó (0)'], [counts[1], counts[0]],
       color=['#2563eb', '#ef4444'], edgecolor='white')
ax.set_title('Desbalance de clases — dataset imputado')
ax.set_ylabel('Número de registros')
for bar, val in zip(ax.patches, [counts[1], counts[0]]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 400,
            f'{val:,}\n({val/len(df_raw)*100:.1f}%)',
            ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(IMG_DIR / '00_desbalance_clases.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Correlación de numéricas con retention ────────────────────────────────────
numeric_df = df_raw.select_dtypes(include='number')
corr = (numeric_df.corr()[TARGET]
        .drop(TARGET)
        .sort_values(key=abs, ascending=False))

fig, ax = plt.subplots(figsize=(7, len(corr)*0.3 + 1))
colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in corr]
ax.barh(corr.index, corr.values, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Correlación de Pearson con retention (dataset imputado)')
ax.set_xlabel('Coeficiente')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(IMG_DIR / '00_correlaciones.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Missingness residual tras imputación ─────────────────────────────────────
# Solo average.first.period (estructural: no existe en PreTec21) debería tener NaN.
nulls = df_raw.isnull().sum()
null_df = (pd.DataFrame({'Nulos': nulls, '%': (nulls / len(df_raw) * 100).round(2)})
           .query('Nulos > 0')
           .sort_values('%', ascending=False))
print(f"Columnas con NaN residual tras imputación: {len(null_df)}")
print(null_df.to_string())

## 4. Ingeniería de Variables y Split Temporal

A diferencia de `exploration.ipynb`, aquí **no** aplicamos imputación de missingness
(ya fue hecha en el pipeline). Las transformaciones que sí realizamos son:

| Paso | Acción |
|------|--------|
| Drop | Columnas con fuga, identificadores, redundancias |
| Norm | `admission.test` PAA/PAL → escala 0–1 (`admission_test_norm`) |
| Enc  | `first.generation` → numérico 0/1 + indicador `first_gen_present` |
| Enc  | `max.degree.parents` → ordinal 0–4 (`educ_padres_max`) |
| Enc  | Categóricas nominales → label/ordinal encode |
| Feat | `apoyo_financiero` = `total.scholarship.loan` |
| Feat | `has_extracurriculars` derivado del indicador `has_life_activities` |
| Split | PreTec21 (AD14–18) = train / Tec21 (AD19–20) = test |

In [ ]:
df = df_raw.copy()

# ── 4.1 Eliminar columnas con fuga o irrelevantes ────────────────────────────
DROP_COLS = [
    'student.id',
    'level',                           # constante: todo Undergraduate
    'average.first.period',            # solo en Tec21 → fuga de datos
    'failed.subject.first.period',     # solo en Tec21 → fuga
    'dropped.subject.first.period',    # solo en Tec21 → fuga
    'dropout.semester',                # derivado de retention → fuga
    'program',                         # alta cardinalidad, irrelevante
    'id.school.origin',                # identificador
    'scholarship.type',                # redundante con scholarship.perc
    'school.cost',                     # redundante con socioeconomic.level
    'parents.exatec', 'father.exatec', 'mother.exatec',
    'father.education.complete', 'father.education.summary',
    'mother.education.complete', 'mother.education.summary',
    'scholarship.perc', 'loan.perc',   # ya en total.scholarship.loan
]
df.drop(columns=[c for c in DROP_COLS if c in df.columns], inplace=True)
print(f"Columnas tras exclusión: {df.shape[1]}  →  {df.columns.tolist()}")

In [ ]:
# ── 4.2 Normalizar admission.test → escala 0–1 ───────────────────────────────
# El dataset imputado ya tiene valores numéricos para admission.test,
# pero la escala sigue siendo PAA (400–1600) o PAL (0–100) según el alumno.
# La normalización unifica ambas escalas.
def norm_admission_test(val):
    """PAA (>100) → (val-400)/1200; PAL (0-100) → val/100. NaN → NaN."""
    if pd.isna(val):
        return np.nan
    try:
        f = float(val)
        if f > 100:
            return max(0.0, (f - 400) / 1200.0)
        elif f >= 0:
            return f / 100.0
    except Exception:
        pass
    return np.nan

df['admission_test_norm'] = df['admission.test'].apply(norm_admission_test)
df.drop(columns=['admission.test'], inplace=True)

print(f"admission_test_norm — NaN: {df['admission_test_norm'].isna().sum()}")
print(f"  media: {df['admission_test_norm'].mean():.3f}  |  std: {df['admission_test_norm'].std():.3f}")

# Imputar residuales con mediana del train (safety imputer)
_train_mask = df['generation'].isin(TRAIN_GENS)
_med_at = df.loc[_train_mask, 'admission_test_norm'].median()
df['admission_test_norm'].fillna(_med_at, inplace=True)
print(f"  → mediana train imputada: {_med_at:.3f}  |  NaN residuales: {df['admission_test_norm'].isna().sum()}")

In [ ]:
# ── 4.3 first.generation → first_gen_enc + conservar first_gen_present ────────
# El pipeline imputó first.generation con RF (valores Yes/No).
# Conservamos el indicador original de missingness (first_gen_present) como
# variable adicional, ya que el patrón MNAR tiene valor predictivo.
def encode_first_gen(val):
    s = str(val).strip() if pd.notna(val) else 'Unknown'
    if s == 'Yes':  return 1
    if s == 'No':   return 0
    return 0  # fallback (no debería ocurrir en dataset imputado)

df['first_gen_enc'] = df['first.generation'].apply(encode_first_gen)
df.drop(columns=['first.generation'], inplace=True)

vals = df['first_gen_enc'].value_counts().sort_index()
print("first_gen_enc — distribución:")
print(f"  0 (No) : {vals.get(0,0):>6,} ({vals.get(0,0)/len(df)*100:.1f}%)")
print(f"  1 (Yes): {vals.get(1,0):>6,} ({vals.get(1,0)/len(df)*100:.1f}%)")
print(f"\nfirst_gen_present (dato original): {df['first_gen_present'].mean()*100:.1f}% tenían el dato")

In [ ]:
# ── 4.4 max.degree.parents → educ_padres_max ordinal ─────────────────────────
EDU_ORD = {
    'No information': 0, 'MISSING': 0,
    'No degree': 1,
    'Undergraduate degree': 2,
    'Master degree': 3,
    'PhD': 4,
}
df['educ_padres_max'] = df['max.degree.parents'].map(EDU_ORD).fillna(0).astype(int)
df.drop(columns=['max.degree.parents'], inplace=True)
print("educ_padres_max — distribución:")
print(df['educ_padres_max'].value_counts().sort_index().to_string())

In [ ]:
# ── 4.5 apoyo_financiero + has_extracurriculars ───────────────────────────────
df.rename(columns={'total.scholarship.loan': 'apoyo_financiero'}, inplace=True)

# has_extracurriculars: reutilizamos el indicador has_life_activities
# (si el alumno tenía datos en total.life.activities = participó)
df['has_extracurriculars'] = df['has_life_activities'].copy()

# Eliminar columnas de actividades detalladas si existen
ACTIVITY_COLS = [
    'physical.education', 'cultural.diffusion', 'student.society',
    'total.life.activities', 'athletic.sports', 'art.culture',
    'student.society.leadership', 'life.work.mentoring', 'wellness.activities'
]
df.drop(columns=[c for c in ACTIVITY_COLS if c in df.columns], inplace=True)
print(f"has_extracurriculars → {df['has_extracurriculars'].sum():,} alumnos con actividades ({df['has_extracurriculars'].mean()*100:.1f}%)")

In [ ]:
# ── 4.6 Encoding de categóricas ──────────────────────────────────────────────
# gender
df['gender_enc'] = df['gender'].map({'Male': 1, 'Female': 0}).fillna(0).astype(int)
df.drop(columns=['gender'], inplace=True)

# tec.no.tec
df['tec_enc'] = df['tec.no.tec'].map({'TEC': 1, 'NO TEC': 0}).fillna(0).astype(int)
df.drop(columns=['tec.no.tec'], inplace=True)

# foreign
df['foreign_enc'] = df['foreign'].map(
    {'Local': 0, 'Yes: National': 1, 'Yes: Foreigner': 2}
).fillna(0).astype(int)
df.drop(columns=['foreign'], inplace=True)

# zone.type — nota: 'MISSING' es categoría válida para filas con missingness estructural
zone_map = {'Rural': 0, 'Semiurban': 1, 'Urban': 2,
            'No information': 1, 'MISSING': 1}  # MISSING → valor neutral
df['zone_enc'] = df['zone.type'].map(zone_map).fillna(1).astype(int)
df.drop(columns=['zone.type'], inplace=True)

# socioeconomic.level
sec_map = {'No information': 0, 'MISSING': 0,
           'Level 1': 1, 'Level 2': 2, 'Level 3': 3,
           'Level 4': 4, 'Level 5': 5, 'Level 6': 6, 'Level 7': 7}
df['socioec_enc'] = df['socioeconomic.level'].map(sec_map).fillna(0).astype(int)
df.drop(columns=['socioeconomic.level'], inplace=True)

# social.lag
lag_map = {'No information': 0, 'MISSING': 0,
           'Low': 1, 'Medium': 2, 'High': 3}
df['social_lag_enc'] = df['social.lag'].map(lag_map).fillna(0).astype(int)
df.drop(columns=['social.lag'], inplace=True)

# region → label encode
le_region = LabelEncoder()
df['region_enc'] = le_region.fit_transform(df['region'].fillna('Unknown'))
df.drop(columns=['region'], inplace=True)

# school → label encode (conservar string para análisis de heterogeneidad)
le_school = LabelEncoder()
df['school_enc'] = le_school.fit_transform(df['school'].fillna('Unknown'))

print("Columnas finales:", df.columns.tolist())
print("Shape final:", df.shape)
print("\nMissing residual:")
missing = df.isnull().sum()
print(missing[missing > 0].to_string() if missing.sum() > 0 else '  ✓ Sin NaN')

In [ ]:
# ── 4.7 Definir FEATURE_COLS y split temporal ────────────────────────────────
# Respecto a exploration.ipynb se agregan:
#   general.math.eval  — ahora imputado (antes descartado por baja cobertura)
#   first_gen_present  — indicador MNAR de primera generación
#   parents_edu_present — indicador de disponibilidad de educación padres
#   took_admission_test — indicador MCAR de examen de admisión
#   has_socioeconomic_data, has_social_lag_data, has_zone_data — indicadores estructurales

FEATURE_COLS = [
    # Variables académicas previas
    'PNA', 'admission_test_norm', 'english.evaluation',
    'admission.rubric', 'general.math.eval',
    # Variables institucionales
    'FTE', 'apoyo_financiero', 'online.test',
    # Variables socioeconómicas y familiares
    'has_extracurriculars', 'first_gen_enc', 'educ_padres_max',
    'socioec_enc', 'social_lag_enc',
    # Indicadores de missingness (información adicional)
    'first_gen_present', 'parents_edu_present', 'took_admission_test',
    'has_socioeconomic_data', 'has_social_lag_data', 'has_zone_data',
    # Demográficas
    'age', 'gender_enc', 'tec_enc', 'foreign_enc',
    'zone_enc', 'region_enc', 'school_enc',
]

# Variables de clustering (perfil socioeconómico/académico previo, sin tec_enc)
CLUSTER_COLS = [
    'PNA', 'admission_test_norm', 'english.evaluation', 'admission.rubric',
    'general.math.eval', 'FTE', 'apoyo_financiero',
    'has_extracurriculars', 'first_gen_enc', 'educ_padres_max',
    'socioec_enc', 'social_lag_enc', 'age', 'gender_enc',
    'foreign_enc', 'zone_enc', 'region_enc', 'school_enc', 'online.test',
]

# Máscaras temporales
tec20_mask = df['generation'].isin(TRAIN_GENS)
tec21_mask = df['generation'].isin(TEST_GENS)

# Arrays para modelos supervisados
X_train = df.loc[tec20_mask, FEATURE_COLS].values.astype(float)
y_train = df.loc[tec20_mask, TARGET].values
X_test  = df.loc[tec21_mask, FEATURE_COLS].values.astype(float)
y_test  = df.loc[tec21_mask, TARGET].values

print(f"Entrenamiento (PreTec21, AD14–18): {X_train.shape[0]:>6,}  —  deserción: {(y_train==0).mean()*100:.1f}%")
print(f"Prueba       (Tec21, AD19–20)   : {X_test.shape[0]:>6,}  —  deserción: {(y_test==0).mean()*100:.1f}%")
print(f"\nFeatures del modelo   : {len(FEATURE_COLS)}")
print(f"Features de clustering: {len(CLUSTER_COLS)}")

# Safety imputer: eliminar cualquier NaN residual
_nan_count = np.isnan(X_train).sum()
if _nan_count > 0:
    print(f"\n⚠ NaN en X_train: {_nan_count}")
    imputer_final = SimpleImputer(strategy='median')
    X_train = imputer_final.fit_transform(X_train)
    X_test  = imputer_final.transform(X_test)
    print("✓ Safety imputer aplicado")
else:
    print("\n✓ X_train sin NaN — no requiere safety imputer")
    # Imputar X_test por si acaso
    imputer_final = SimpleImputer(strategy='median')
    X_train = imputer_final.fit_transform(X_train)
    X_test  = imputer_final.transform(X_test)

## 5. K-Means Clustering

**Objetivo:** identificar perfiles latentes de riesgo sin etiquetas supervisadas y
evaluar si esos perfiles persisten entre PreTec21 y Tec21 (invarianza estructural).

**Protocolo:**
1. Estandarizar variables con escalador ajustado **solo** en PreTec21.
2. Determinar k óptimo: método del codo + Silhouette + Davies-Bouldin.
3. Ajustar K-Means en PreTec21; proyectar Tec21 con `predict` (sin reentrenar).
4. Comparar distribución de clusters y perfiles (z-scores) entre regímenes.
5. Reportar invarianza: variación < 15% en z-score por variable clave.

**Criterios de éxito (Entrega 2):**
- Silhouette Score ≥ 0.30
- Davies-Bouldin ≤ 1.50
- Variación de z-score entre regímenes < 20% en variables principales

In [ ]:
# ── 5.1 Preparar datos de clustering ─────────────────────────────────────────
X_clust_train_df = df.loc[tec20_mask, CLUSTER_COLS].copy()
X_clust_test_df  = df.loc[tec21_mask, CLUSTER_COLS].copy()

# Imputar con mediana del train (nunca del test)
clust_medians = X_clust_train_df.median()
X_clust_train_df.fillna(clust_medians, inplace=True)
X_clust_test_df.fillna(clust_medians, inplace=True)

# Escalar con estadísticos del train
scaler_clust  = StandardScaler()
X_clust_train = scaler_clust.fit_transform(X_clust_train_df.values.astype(float))
X_clust_test  = scaler_clust.transform(X_clust_test_df.values.astype(float))

print(f"Train clustering shape: {X_clust_train.shape}")
print(f"Test  clustering shape: {X_clust_test.shape}")

In [ ]:
# ── 5.2 Selección de k óptimo (submuestra de PreTec21) ───────────────────────
inertias, silhouettes, db_scores = [], [], []
K_RANGE = range(2, 9)

rng     = np.random.default_rng(SEED)
idx_sub = rng.choice(len(X_clust_train),
                     size=min(10_000, len(X_clust_train)), replace=False)
X_sub   = X_clust_train[idx_sub]

for k in K_RANGE:
    km   = KMeans(n_clusters=k, n_init=10, random_state=SEED)
    lbls = km.fit_predict(X_sub)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_sub, lbls, sample_size=3_000, random_state=SEED))
    db_scores.append(davies_bouldin_score(X_sub, lbls))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
kl = list(K_RANGE)

axes[0].plot(kl, inertias, 'o-', color='#2563eb')
axes[0].axvline(K_CLUSTERS, color='red', linestyle='--', alpha=0.7, label=f'k={K_CLUSTERS}')
axes[0].set_title('Método del Codo (Inercia)'); axes[0].set_xlabel('k'); axes[0].legend()

axes[1].plot(kl, silhouettes, 'o-', color='#059669')
axes[1].axvline(K_CLUSTERS, color='red', linestyle='--', alpha=0.7)
axes[1].axhline(0.3, color='orange', linestyle=':', alpha=0.7, label='umbral 0.30')
axes[1].set_title('Coeficiente de Silhouette'); axes[1].set_xlabel('k'); axes[1].legend()

axes[2].plot(kl, db_scores, 'o-', color='#d97706')
axes[2].axvline(K_CLUSTERS, color='red', linestyle='--', alpha=0.7)
axes[2].axhline(1.5, color='orange', linestyle=':', alpha=0.7, label='umbral 1.50')
axes[2].set_title('Índice Davies-Bouldin'); axes[2].set_xlabel('k'); axes[2].legend()

plt.suptitle(f'Selección de k óptimo — submuestra 10K sobre PreTec21',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(IMG_DIR / '01_kmeans_seleccion_k.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nk={K_CLUSTERS} — Silhouette: {silhouettes[K_CLUSTERS-2]:.3f} | Davies-Bouldin: {db_scores[K_CLUSTERS-2]:.3f}")
print(f"Criterio Silhouette ≥ 0.30: {'✓ CUMPLE' if silhouettes[K_CLUSTERS-2] >= 0.30 else '✗ No cumple'}")
print(f"Criterio DB ≤ 1.50       : {'✓ CUMPLE' if db_scores[K_CLUSTERS-2] <= 1.50 else '✗ No cumple'}")

In [ ]:
# ── 5.3 Entrenar K-Means en PreTec21 ─────────────────────────────────────────
km_final = KMeans(n_clusters=K_CLUSTERS, n_init=20, random_state=SEED)
km_final.fit(X_clust_train)
labels_train = km_final.labels_

df_train_profiles = df.loc[tec20_mask].copy()
df_train_profiles['cluster'] = labels_train

cluster_stats_train = (
    df_train_profiles
    .groupby('cluster')
    .agg(n=('retention', 'count'),
         dropout_rate=('retention', lambda x: (x == 0).mean()))
    .round(3)
)
cluster_stats_train['pct_total'] = (
    cluster_stats_train['n'] / len(df_train_profiles) * 100
).round(1)
cluster_stats_train = cluster_stats_train.sort_values('dropout_rate', ascending=False)

print("═══ Perfiles PreTec21 ═══")
print(cluster_stats_train.to_string())

# Heatmap de centroides
centers_df = pd.DataFrame(km_final.cluster_centers_, columns=CLUSTER_COLS)
centers_df.index = [f'C{i}' for i in range(K_CLUSTERS)]

fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(centers_df.values, cmap='RdBu_r', aspect='auto', vmin=-2, vmax=2)
ax.set_xticks(range(len(CLUSTER_COLS)))
ax.set_xticklabels(CLUSTER_COLS, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(K_CLUSTERS))
ax.set_yticklabels([f'C{i} (dropout={cluster_stats_train.loc[i, "dropout_rate"]:.2f})'
                    for i in range(K_CLUSTERS)])
plt.colorbar(im, ax=ax, label='z-score')
ax.set_title('Perfiles de clusters — PreTec21 (z-scores)', fontsize=12)
for i in range(K_CLUSTERS):
    for j in range(len(CLUSTER_COLS)):
        v = centers_df.values[i, j]
        ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=6,
                color='white' if abs(v) > 1.2 else 'black')
plt.tight_layout()
plt.savefig(IMG_DIR / '01_kmeans_perfiles_pretec21.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 5.4 Proyectar centroides PreTec21 → Tec21 con predict ────────────────────
labels_test = km_final.predict(X_clust_test)

df_test_profiles = df.loc[tec21_mask].copy()
df_test_profiles['cluster'] = labels_test

cluster_stats_test = (
    df_test_profiles
    .groupby('cluster')
    .agg(n=('retention', 'count'),
         dropout_rate=('retention', lambda x: (x == 0).mean()))
    .round(3)
)
cluster_stats_test['pct_total'] = (
    cluster_stats_test['n'] / len(df_test_profiles) * 100
).round(1)
cluster_stats_test = cluster_stats_test.sort_values('dropout_rate', ascending=False)

print("═══ Perfiles Tec21 (centroides de PreTec21) ═══")
print(cluster_stats_test.to_string())

In [ ]:
# ── 5.5 Comparación y análisis de invarianza ──────────────────────────────────
comp = pd.DataFrame({
    'PreTec21_%'     : cluster_stats_train['pct_total'],
    'Tec21_%'        : cluster_stats_test['pct_total'],
    'PreTec21_dropout': cluster_stats_train['dropout_rate'],
    'Tec21_dropout'  : cluster_stats_test['dropout_rate'],
}).round(3)
comp['Δ_pct']     = (comp['Tec21_%']     - comp['PreTec21_%']).round(1)
comp['Δ_dropout'] = (comp['Tec21_dropout'] - comp['PreTec21_dropout']).round(3)
print("Distribución comparativa PreTec21 vs Tec21:")
print(comp.to_string())

# Drift de centroides
centers_test_emp = np.array([
    X_clust_test[labels_test == k].mean(axis=0)
    for k in range(K_CLUSTERS)
])

diff = np.abs(km_final.cluster_centers_ - centers_test_emp)
diff_df = pd.DataFrame(diff, columns=CLUSTER_COLS,
                        index=[f'C{i}' for i in range(K_CLUSTERS)])

max_diff = diff_df.max(axis=0)
DRIFT_THRESHOLD = 0.15
invariant  = (max_diff < DRIFT_THRESHOLD).sum()
drifted    = len(CLUSTER_COLS) - invariant

print(f"\nVariables con drift Δz ≥ {DRIFT_THRESHOLD}:")
drifted_vars = max_diff[max_diff >= DRIFT_THRESHOLD].sort_values(ascending=False)
print(drifted_vars.to_string() if len(drifted_vars) > 0 else '  Ninguna')
print(f"\n✓ Variables invariantes (Δz < {DRIFT_THRESHOLD}): {invariant}/{len(CLUSTER_COLS)}")
print(f"✗ Variables con drift              : {drifted}/{len(CLUSTER_COLS)}")
print(f"Criterio variación < 20%: {'✓ CUMPLE' if drifted/len(CLUSTER_COLS) < 0.20 else '✗ No cumple'}")

# Heatmap comparativo
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, data, title in [
    (axes[0], km_final.cluster_centers_, 'Centroides aprendidos — PreTec21'),
    (axes[1], centers_test_emp,          'Centroides empíricos — Tec21')
]:
    im = ax.imshow(data, cmap='RdBu_r', aspect='auto', vmin=-2, vmax=2)
    ax.set_xticks(range(len(CLUSTER_COLS)))
    ax.set_xticklabels(CLUSTER_COLS, rotation=45, ha='right', fontsize=7)
    ax.set_yticks(range(K_CLUSTERS))
    ax.set_yticklabels([f'C{i}' for i in range(K_CLUSTERS)])
    ax.set_title(title)
    plt.colorbar(im, ax=ax, label='z-score')

plt.suptitle('Estabilidad de perfiles entre regímenes educativos',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(IMG_DIR / '01_kmeans_invarianza.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 5.6 Preparar datos para RF y Regresión Logística ─────────────────────────
FEATURE_COLS_RF = [c for c in FEATURE_COLS if c != 'tec_enc']

X_rf_lr_train = df_train_profiles[FEATURE_COLS_RF].values.astype(float)
y_rf_lr_train = df_train_profiles['cluster'].values
X_rf_lr_test  = df_test_profiles[FEATURE_COLS_RF].values.astype(float)
y_rf_lr_test  = df_test_profiles['cluster'].values

imputer_rf = SimpleImputer(strategy='median')
X_rf_lr_train = imputer_rf.fit_transform(X_rf_lr_train)
X_rf_lr_test  = imputer_rf.transform(X_rf_lr_test)

print(f"X_rf_lr_train: {X_rf_lr_train.shape}")
print(f"X_rf_lr_test : {X_rf_lr_test.shape}")
print("\nDistribución de clusters en PreTec21:")
for c, n in sorted(zip(*np.unique(y_rf_lr_train, return_counts=True))):
    print(f"  Cluster {c}: {n:>5,}  ({n/len(y_rf_lr_train)*100:.1f}%)")

# Objetos disponibles para etapas siguientes
print("\n─── Objetos disponibles ───")
print("  km_final, scaler_clust, df_train_profiles, df_test_profiles")
print("  X_rf_lr_train / y_rf_lr_train, X_rf_lr_test / y_rf_lr_test")

## 6. Regresión Logística

**Rol:** línea base interpretable y herramienta de comparación entre regímenes.
Los coeficientes con IC permiten cuantificar directamente si el efecto de cada
predictor cambia entre PreTec21 y Tec21.

**Criterio de éxito:** AUC-ROC ≥ 0.60 en Tec21 (métrica de control).

In [ ]:
# ── 6.1 Estandarizar y K-Fold CV ─────────────────────────────────────────────
scaler_model = StandardScaler()
X_train_sc   = scaler_model.fit_transform(X_train)
X_test_sc    = scaler_model.transform(X_test)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
lr  = LogisticRegression(class_weight='balanced', max_iter=2000,
                          random_state=SEED, solver='lbfgs')

cv_lr = cross_validate(
    lr, X_train_sc, y_train,
    cv=skf,
    scoring=['roc_auc', 'recall', 'f1', 'precision'],
    return_train_score=True, n_jobs=-1
)
print("Regresión Logística — K-Fold CV (PreTec21):")
print(f"  AUC-ROC  : {cv_lr['test_roc_auc'].mean():.3f} ± {cv_lr['test_roc_auc'].std():.3f}")
print(f"  Recall   : {cv_lr['test_recall'].mean():.3f} ± {cv_lr['test_recall'].std():.3f}")
print(f"  F1-Score : {cv_lr['test_f1'].mean():.3f} ± {cv_lr['test_f1'].std():.3f}")
print(f"  Precision: {cv_lr['test_precision'].mean():.3f} ± {cv_lr['test_precision'].std():.3f}")

In [ ]:
# ── 6.2 Curva de aprendizaje ──────────────────────────────────────────────────
train_sizes, tr_scores, val_scores = learning_curve(
    lr, X_train_sc, y_train,
    train_sizes=np.linspace(0.1, 1.0, 8),
    cv=skf, scoring='roc_auc', n_jobs=-1
)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(train_sizes, tr_scores.mean(1),  'o-', color='#2563eb', label='Entrenamiento')
ax.plot(train_sizes, val_scores.mean(1), 'o-', color='#ef4444', label='Validación CV')
ax.fill_between(train_sizes,
    tr_scores.mean(1)-tr_scores.std(1), tr_scores.mean(1)+tr_scores.std(1),
    alpha=0.1, color='#2563eb')
ax.fill_between(train_sizes,
    val_scores.mean(1)-val_scores.std(1), val_scores.mean(1)+val_scores.std(1),
    alpha=0.1, color='#ef4444')
ax.axhline(0.6, color='gray', linestyle='--', alpha=0.6, label='Umbral AUC 0.60')
ax.set_xlabel('Tamaño entrenamiento'); ax.set_ylabel('AUC-ROC')
ax.set_title('Curva de aprendizaje — Regresión Logística')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(IMG_DIR / '02_lr_learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 6.3 Umbral óptimo (OOF) + modelo final en Tec21 ──────────────────────────
oof_proba_lr = np.zeros(len(X_train_sc))
for tr_idx, val_idx in skf.split(X_train_sc, y_train):
    lr.fit(X_train_sc[tr_idx], y_train[tr_idx])
    oof_proba_lr[val_idx] = lr.predict_proba(X_train_sc[val_idx])[:, 1]

prec_oof, rec_oof, thr_oof = precision_recall_curve(y_train, oof_proba_lr)
f1_oof    = 2 * prec_oof * rec_oof / (prec_oof + rec_oof + 1e-8)
best_thr_lr = thr_oof[np.argmax(f1_oof[:-1])]
print(f"Umbral óptimo (OOF, max-F1): {best_thr_lr:.3f}")

# Modelo final
lr_final = LogisticRegression(class_weight='balanced', max_iter=2000,
                               random_state=SEED, solver='lbfgs')
lr_final.fit(X_train_sc, y_train)
y_proba_lr = lr_final.predict_proba(X_test_sc)[:, 1]
y_pred_lr  = (y_proba_lr >= best_thr_lr).astype(int)

# Bootstrap IC 95%
rng_bs = np.random.default_rng(SEED)
n_test = len(y_test)
boot_aucs_lr, boot_rec_lr, boot_f1_lr = [], [], []
for _ in range(N_BOOTSTRAP):
    idx = rng_bs.integers(0, n_test, n_test)
    if len(np.unique(y_test[idx])) < 2: continue
    boot_aucs_lr.append(roc_auc_score(y_test[idx], y_proba_lr[idx]))
    yb = (y_proba_lr[idx] >= best_thr_lr).astype(int)
    boot_rec_lr.append(recall_score(y_test[idx], yb, zero_division=0))
    boot_f1_lr.append(f1_score(y_test[idx], yb, zero_division=0))

def ci(arr): return np.percentile(arr, [2.5, 97.5])

auc_lr  = roc_auc_score(y_test, y_proba_lr)
rec_lr  = recall_score(y_test, y_pred_lr)
f1_lr   = f1_score(y_test, y_pred_lr)
prec_lr = precision_score(y_test, y_pred_lr)

print(f"\n─── Resultados en Tec21 (umbral={best_thr_lr:.2f}) ───")
print(f"  AUC-ROC  : {auc_lr:.3f}  IC95% [{ci(boot_aucs_lr)[0]:.3f}–{ci(boot_aucs_lr)[1]:.3f}]")
print(f"  Recall   : {rec_lr:.3f}  IC95% [{ci(boot_rec_lr)[0]:.3f}–{ci(boot_rec_lr)[1]:.3f}]")
print(f"  F1-Score : {f1_lr:.3f}  IC95% [{ci(boot_f1_lr)[0]:.3f}–{ci(boot_f1_lr)[1]:.3f}]")
print(f"  Precision: {prec_lr:.3f}")
print(f"\nCriterio AUC ≥ 0.60: {'✓ CUMPLE' if auc_lr >= 0.60 else '✗ No cumple'}")

In [ ]:
# ── 6.4 Curvas ROC y PR + Matriz de confusión ─────────────────────────────────
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_proba_lr)
prec_lr_c, rec_lr_c, _ = precision_recall_curve(y_test, y_proba_lr)
cm_lr = confusion_matrix(y_test, y_pred_lr)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(fpr_lr, tpr_lr, lw=2, color='#2563eb', label=f'LR (AUC={auc_lr:.3f})')
axes[0].plot([0,1],[0,1],'--', color='gray', alpha=0.5)
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('Curva ROC — LR (Tec21)'); axes[0].legend(); axes[0].grid(alpha=0.3)

no_skill = (y_test == 0).mean()
axes[1].plot(rec_lr_c, prec_lr_c, lw=2, color='#059669')
axes[1].axhline(no_skill, linestyle='--', color='gray', alpha=0.5, label=f'No-skill ({no_skill:.2f})')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Curva PR — LR (Tec21)'); axes[1].legend(); axes[1].grid(alpha=0.3)

ConfusionMatrixDisplay(cm_lr, display_labels=['Desertó','Retuvo']).plot(
    ax=axes[2], colorbar=False, cmap='Blues')
axes[2].set_title(f'Matriz de Confusión LR (umbral={best_thr_lr:.2f})')

plt.tight_layout()
plt.savefig(IMG_DIR / '02_lr_evaluacion.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 6.5 Coeficientes (odds ratios) ────────────────────────────────────────────
coefs    = lr_final.coef_[0]
odds     = np.exp(coefs)
coef_df  = pd.DataFrame({
    'Variable'   : FEATURE_COLS,
    'Coeficiente': coefs,
    'Odds Ratio' : odds,
}).sort_values('Coeficiente', key=abs, ascending=False)

print("Top 10 variables por magnitud de coeficiente:")
print(coef_df.head(10)[['Variable','Coeficiente','Odds Ratio']].to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 6))
top15   = coef_df.head(15)
colors  = ['#2563eb' if c > 0 else '#ef4444' for c in top15['Coeficiente']]
ax.barh(top15['Variable'], top15['Coeficiente'], color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Coeficientes — Regresión Logística (Top 15)')
ax.set_xlabel('Coeficiente (log-odds) — Positivo = más retención')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(IMG_DIR / '02_lr_coeficientes.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Random Forest

**Rol:** capturar interacciones no lineales entre predictores (ej. NSE × primera generación)
que la regresión logística no modela sin términos de interacción explícitos.
La importancia de variables valida si las mismas features que el clustering identifica
como definitorias de riesgo también emergen como las más importantes en el RF.

In [ ]:
# ── 7.1 Búsqueda de hiperparámetros (RandomizedSearchCV) ─────────────────────
param_dist = {
    'n_estimators'    : randint(100, 400),
    'max_depth'       : [5, 8, 12, 15, None],
    'min_samples_leaf': randint(5, 40),
    'max_features'    : ['sqrt', 'log2', 0.4],
    'class_weight'    : ['balanced', 'balanced_subsample'],
}

rf_base = RandomForestClassifier(random_state=SEED, n_jobs=1)
rscv = RandomizedSearchCV(
    rf_base, param_dist,
    n_iter=25, cv=skf,
    scoring='roc_auc',
    random_state=SEED, n_jobs=-1, verbose=1
)
print("Iniciando RandomizedSearchCV (n_iter=25, cv=5)...")
t0 = time.time()
rscv.fit(X_train_sc, y_train)
print(f"Completado en {(time.time()-t0)/60:.1f} min")
print(f"\nMejores hiperparámetros:")
for k, v in rscv.best_params_.items(): print(f"  {k}: {v}")
print(f"\nMejor AUC-ROC en CV: {rscv.best_score_:.4f}")

In [ ]:
# ── 7.2 Umbral óptimo (OOF) + métricas en Tec21 ──────────────────────────────
oof_proba_rf = np.zeros(len(X_train_sc))
rf_oof = RandomForestClassifier(**rscv.best_params_, random_state=SEED, n_jobs=1)
for tr_idx, val_idx in skf.split(X_train_sc, y_train):
    rf_oof.fit(X_train_sc[tr_idx], y_train[tr_idx])
    oof_proba_rf[val_idx] = rf_oof.predict_proba(X_train_sc[val_idx])[:, 1]

prec_oof_rf, rec_oof_rf, thr_oof_rf = precision_recall_curve(y_train, oof_proba_rf)
f1_oof_rf  = 2 * prec_oof_rf * rec_oof_rf / (prec_oof_rf + rec_oof_rf + 1e-8)
best_thr_rf = thr_oof_rf[np.argmax(f1_oof_rf[:-1])]
print(f"Umbral óptimo RF (OOF): {best_thr_rf:.3f}")

# Modelo final
rf_final = rscv.best_estimator_
rf_final.fit(X_train_sc, y_train)
y_proba_rf = rf_final.predict_proba(X_test_sc)[:, 1]
y_pred_rf  = (y_proba_rf >= best_thr_rf).astype(int)

# Bootstrap IC 95%
boot_aucs_rf, boot_rec_rf, boot_f1_rf = [], [], []
for _ in range(N_BOOTSTRAP):
    idx = rng_bs.integers(0, n_test, n_test)
    if len(np.unique(y_test[idx])) < 2: continue
    boot_aucs_rf.append(roc_auc_score(y_test[idx], y_proba_rf[idx]))
    yb = (y_proba_rf[idx] >= best_thr_rf).astype(int)
    boot_rec_rf.append(recall_score(y_test[idx], yb, zero_division=0))
    boot_f1_rf.append(f1_score(y_test[idx], yb, zero_division=0))

auc_rf  = roc_auc_score(y_test, y_proba_rf)
rec_rf  = recall_score(y_test, y_pred_rf)
f1_rf   = f1_score(y_test, y_pred_rf)
prec_rf = precision_score(y_test, y_pred_rf)

print(f"\n─── Resultados en Tec21 (umbral={best_thr_rf:.2f}) ───")
print(f"  AUC-ROC  : {auc_rf:.3f}  IC95% [{ci(boot_aucs_rf)[0]:.3f}–{ci(boot_aucs_rf)[1]:.3f}]")
print(f"  Recall   : {rec_rf:.3f}  IC95% [{ci(boot_rec_rf)[0]:.3f}–{ci(boot_rec_rf)[1]:.3f}]")
print(f"  F1-Score : {f1_rf:.3f}  IC95% [{ci(boot_f1_rf)[0]:.3f}–{ci(boot_f1_rf)[1]:.3f}]")
print(f"  Precision: {prec_rf:.3f}")
print(f"\nCriterio AUC ≥ 0.60: {'✓ CUMPLE' if auc_rf >= 0.60 else '✗ No cumple'}")

In [ ]:
# ── 7.3 Curvas ROC y PR + Matriz de confusión ─────────────────────────────────
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_proba_rf)
prec_rf_c, rec_rf_c, _ = precision_recall_curve(y_test, y_proba_rf)
cm_rf = confusion_matrix(y_test, y_pred_rf)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(fpr_lr, tpr_lr, lw=2, color='#2563eb', label=f'LR (AUC={auc_lr:.3f})')
axes[0].plot(fpr_rf, tpr_rf, lw=2, color='#d97706', label=f'RF (AUC={auc_rf:.3f})')
axes[0].plot([0,1],[0,1],'--', color='gray', alpha=0.5)
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('ROC — LR vs RF (Tec21)'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(rec_lr_c, prec_lr_c, lw=2, color='#2563eb', label='LR')
axes[1].plot(rec_rf_c, prec_rf_c, lw=2, color='#d97706', label='RF')
axes[1].axhline(no_skill, linestyle='--', color='gray', alpha=0.5, label='No-skill')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall — LR vs RF (Tec21)'); axes[1].legend(); axes[1].grid(alpha=0.3)

ConfusionMatrixDisplay(cm_rf, display_labels=['Desertó','Retuvo']).plot(
    ax=axes[2], colorbar=False, cmap='Oranges')
axes[2].set_title(f'Conf. Matrix RF (umbral={best_thr_rf:.2f})')

plt.tight_layout()
plt.savefig(IMG_DIR / '03_rf_evaluacion.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7.4 Importancia de variables (Gini + Permutación) ────────────────────────
gini_imp = pd.Series(rf_final.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)

print("Calculando permutation importance (~1 min)...")
perm = permutation_importance(
    rf_final, X_test_sc, y_test,
    n_repeats=15, random_state=SEED, scoring='roc_auc', n_jobs=-1
)
perm_imp = pd.Series(perm.importances_mean, index=FEATURE_COLS).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
top_n = 15

gini_top = gini_imp.head(top_n)
axes[0].barh(gini_top.index[::-1], gini_top.values[::-1], color='#d97706', edgecolor='white')
axes[0].set_title(f'Importancia Gini — Top {top_n}')
axes[0].set_xlabel('Importancia (Gini)')

perm_top = perm_imp.head(top_n)
perm_order = perm_imp.argsort()[::-1][:top_n]
axes[1].barh(perm_top.index[::-1], perm_top.values[::-1],
             xerr=perm.importances_std[perm_order][::-1],
             color='#059669', edgecolor='white', capsize=4)
axes[1].set_title(f'Importancia Permutación — Top {top_n}')
axes[1].set_xlabel('Δ AUC-ROC')
axes[1].axvline(0, color='black', linewidth=0.8)

plt.suptitle('Importancia de Variables — Random Forest', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(IMG_DIR / '03_rf_importancia.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTop 10 — Gini:")
print(gini_imp.head(10).to_string())
print("\nTop 10 — Permutación:")
print(perm_imp.head(10).to_string())

## 8. Comparación Sistemática

El criterio de selección no es el AUC más alto, sino la capacidad de identificar
variables invariantes e interpretables entre regímenes educativos.

In [ ]:
# ── 8.1 Tabla comparativa ─────────────────────────────────────────────────────
def fmt_ci(vals):
    lo, hi = ci(vals)
    return f"[{lo:.3f}–{hi:.3f}]"

summary = {
    'Modelo'               : ['Naive (baseline)', 'Regresión Logística', 'Random Forest'],
    'AUC-ROC'              : ['0.500', f'{auc_lr:.3f}', f'{auc_rf:.3f}'],
    'IC 95% AUC'           : ['—', fmt_ci(boot_aucs_lr), fmt_ci(boot_aucs_rf)],
    'Recall'               : ['0.000', f'{rec_lr:.3f}', f'{rec_rf:.3f}'],
    'IC 95% Recall'        : ['—', fmt_ci(boot_rec_lr), fmt_ci(boot_rec_rf)],
    'F1-Score'             : ['0.000', f'{f1_lr:.3f}', f'{f1_rf:.3f}'],
    'IC 95% F1'            : ['—', fmt_ci(boot_f1_lr), fmt_ci(boot_f1_rf)],
    'Precision'            : ['—', f'{prec_lr:.3f}', f'{prec_rf:.3f}'],
    'Interpretabilidad'    : ['Alta (trivial)', 'Alta (coefs.)', 'Media (importancias)'],
    'Responde preg. invest.': ['No', 'Sí (coefs. comparables)', 'Parcial (importancias)'],
}
df_summary = pd.DataFrame(summary)
print("Tabla comparativa — Sección 8")
display(df_summary.set_index('Modelo'))

In [ ]:
# ── 8.2 Superposición de curvas ROC ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr_lr, tpr_lr, lw=2.5, color='#2563eb', label=f'Regresión Logística  AUC={auc_lr:.3f}')
ax.plot(fpr_rf, tpr_rf, lw=2.5, color='#d97706', label=f'Random Forest        AUC={auc_rf:.3f}')
ax.plot([0,1],[0,1],'--', color='#94a3b8', lw=1.2, label='Aleatorio  AUC=0.500')
ax.fill_between(fpr_lr, tpr_lr, alpha=0.05, color='#2563eb')
ax.fill_between(fpr_rf, tpr_rf, alpha=0.05, color='#d97706')
ax.set_xlabel('Tasa de Falsos Positivos (1-Especificidad)', fontsize=12)
ax.set_ylabel('Tasa de Verdaderos Positivos (Sensibilidad)', fontsize=12)
ax.set_title('Curvas ROC — comparación en Tec21', fontsize=13, fontweight='bold')
ax.legend(fontsize=11); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(IMG_DIR / '04_comparacion_roc.png', dpi=150, bbox_inches='tight')
plt.show()

print(
    "\nSelección de modelo para Componente 3:\n"
    "La Regresión Logística se selecciona como modelo principal para el análisis de\n"
    "invarianza porque:\n"
    "  (1) Sus coeficientes permiten comparación directa entre PreTec21 y Tec21.\n"
    "  (2) El AUC supera el umbral de 0.60 establecido en el diseño experimental.\n"
    "  (3) Random Forest ofrece mayor AUC pero menor interpretabilidad directa;\n"
    "      se mantiene como modelo complementario para validación por importancia de variables.\n"
)

## 9. Experimentos Negativos

*"Reportar honestamente qué NO funcionó demuestra rigor científico y evita que otros
repitan experimentos infructuosos."*

Se documentan tres experimentos negativos surgidos del proceso de experimentación.

In [ ]:
# ── 9.1 Experimento Negativo 1 — PCA para reducción de dimensionalidad ────────
print("=== Experimento Negativo 1: PCA ===")
print("Hipótesis: k=8 componentes PCA mantienen AUC y simplifican el modelo.\n")

results_pca = {}
for n_comp in [5, 8, 12, len(FEATURE_COLS)]:
    if n_comp == len(FEATURE_COLS):
        X_tr_pca, X_te_pca = X_train_sc, X_test_sc
        label = 'Sin PCA (baseline)'
    else:
        pca = PCA(n_components=n_comp, random_state=SEED)
        X_tr_pca = pca.fit_transform(X_train_sc)
        X_te_pca = pca.transform(X_test_sc)
        var_exp  = pca.explained_variance_ratio_.sum()
        label    = f'PCA k={n_comp} ({var_exp*100:.0f}% var.)'

    lr_pca = LogisticRegression(class_weight='balanced', max_iter=2000, random_state=SEED)
    lr_pca.fit(X_tr_pca, y_train)
    auc_pca = roc_auc_score(y_test, lr_pca.predict_proba(X_te_pca)[:, 1])
    results_pca[label] = auc_pca
    print(f"  {label:<40} → AUC = {auc_pca:.4f}")

baseline_auc = results_pca['Sin PCA (baseline)']
k8_auc = list(results_pca.values())[1]
print(f"\n✗ CONCLUSIÓN: PCA k=8 reduce el AUC en {baseline_auc - k8_auc:.4f} puntos")
print("  y elimina la interpretabilidad directa de los coeficientes por variable.")
print("  DECISIÓN: se descarta PCA en el modelo principal.")

In [ ]:
# ── 9.2 Experimento Negativo 2 — Split aleatorio vs. split temporal ───────────
print("=== Experimento Negativo 2: Split aleatorio vs. split temporal ===")
print("Hipótesis: split aleatorio 80/20 da resultados equivalentes al temporal.\n")

X_all = df[FEATURE_COLS].fillna(0).values.astype(float)
y_all = df[TARGET].values

X_tr_rand, X_te_rand, y_tr_rand, y_te_rand = train_test_split(
    X_all, y_all, test_size=0.2, random_state=SEED, stratify=y_all
)
scaler_rand  = StandardScaler()
X_tr_rand_sc = scaler_rand.fit_transform(X_tr_rand)
X_te_rand_sc = scaler_rand.transform(X_te_rand)

lr_rand = LogisticRegression(class_weight='balanced', max_iter=2000, random_state=SEED)
lr_rand.fit(X_tr_rand_sc, y_tr_rand)
auc_rand = roc_auc_score(y_te_rand, lr_rand.predict_proba(X_te_rand_sc)[:, 1])

print(f"  AUC — Split temporal (PreTec21→Tec21): {auc_lr:.4f}")
print(f"  AUC — Split aleatorio (80/20)        : {auc_rand:.4f}")
print(f"  Diferencia                            : +{auc_rand - auc_lr:.4f}")
print(f"\n✗ CONCLUSIÓN: el split aleatorio infla el AUC ~{(auc_rand-auc_lr):.3f} puntos al")
print("  mezclar cohortes de ambos modelos educativos en entrenamiento y prueba.")
print("  Esto NO responde la pregunta de invarianza. DECISIÓN: se mantiene split temporal.")

In [ ]:
# ── 9.3 Experimento Negativo 3 — Sin ajuste de pesos de clase ────────────────
print("=== Experimento Negativo 3: Sin class_weight ===")
print("Hipótesis: dado el N grande (~77K), omitir class_weight es viable.\n")

lr_noweight = LogisticRegression(class_weight=None, max_iter=2000, random_state=SEED)
lr_noweight.fit(X_train_sc, y_train)
y_proba_nw = lr_noweight.predict_proba(X_test_sc)[:, 1]
y_pred_nw  = (y_proba_nw >= 0.5).astype(int)

auc_nw  = roc_auc_score(y_test, y_proba_nw)
rec_nw  = recall_score(y_test, y_pred_nw)
f1_nw   = f1_score(y_test, y_pred_nw)
prec_nw = precision_score(y_test, y_pred_nw, zero_division=0)

print(f"  Con class_weight='balanced' (seleccionado):")
print(f"    AUC={auc_lr:.3f}  Recall={rec_lr:.3f}  F1={f1_lr:.3f}  Precision={prec_lr:.3f}")
print(f"  Sin class_weight (None, umbral=0.5):")
print(f"    AUC={auc_nw:.3f}  Recall={rec_nw:.3f}  F1={f1_nw:.3f}  Precision={prec_nw:.3f}")
print(f"\n  Predicciones únicas: {np.unique(y_pred_nw, return_counts=True)}")
print(f"\n✗ CONCLUSIÓN: sin ajuste de pesos el modelo predice casi siempre retención.")
print(f"  Recall = {rec_nw:.3f} → prácticamente no identifica desertores.")
print("  DECISIÓN: class_weight='balanced' + threshold moving es obligatorio.")

## 10. Resumen Final y Guardado de Resultados

In [ ]:
# ── Guardar tabla de resultados ───────────────────────────────────────────────
results_path = DATA_DIR / 'resultados_experimentacion.csv'
df_results = pd.DataFrame({
    'modelo'    : ['Naive', 'Logistic Regression', 'Random Forest'],
    'auc_roc'   : [0.5,    auc_lr,               auc_rf],
    'recall'    : [0.0,    rec_lr,               rec_rf],
    'f1'        : [0.0,    f1_lr,                f1_rf],
    'precision' : [0.0,    prec_lr,              prec_rf],
    'auc_ci_low': [0.5,    ci(boot_aucs_lr)[0],  ci(boot_aucs_rf)[0]],
    'auc_ci_high':[0.5,    ci(boot_aucs_lr)[1],  ci(boot_aucs_rf)[1]],
    'threshold' : [0.5,    best_thr_lr,          best_thr_rf],
})
df_results.to_csv(results_path, index=False)
print(f"✓ Resultados guardados en: {results_path}")

print("\n─── Resumen de métricas ──────────────────────────────────────────────")
print(df_results.to_string(index=False))

print("\n─── K-Means: invarianza ─────────────────────────────────────────────")
print(f"  Variables invariantes (Δz < {DRIFT_THRESHOLD}): {invariant}/{len(CLUSTER_COLS)}")
print(f"  Silhouette k={K_CLUSTERS}: {silhouettes[K_CLUSTERS-2]:.3f}")
print(f"  Davies-Bouldin k={K_CLUSTERS}: {db_scores[K_CLUSTERS-2]:.3f}")

print("\n─── Criterios de éxito ──────────────────────────────────────────────")
print(f"  Silhouette ≥ 0.30     : {'✓' if silhouettes[K_CLUSTERS-2] >= 0.30 else '✗'}  ({silhouettes[K_CLUSTERS-2]:.3f})")
print(f"  Davies-Bouldin ≤ 1.50 : {'✓' if db_scores[K_CLUSTERS-2] <= 1.50 else '✗'}  ({db_scores[K_CLUSTERS-2]:.3f})")
print(f"  AUC-ROC LR ≥ 0.60    : {'✓' if auc_lr >= 0.60 else '✗'}  ({auc_lr:.3f})")
print(f"  AUC-ROC RF ≥ 0.60    : {'✓' if auc_rf >= 0.60 else '✗'}  ({auc_rf:.3f})")
print(f"  Drift clusters < 20%  : {'✓' if drifted/len(CLUSTER_COLS) < 0.20 else '✗'}  ({drifted/len(CLUSTER_COLS)*100:.0f}% drifted)")